In [1]:
import sys, yaml
sys.path.append("../..")

import numpy as np
import torch

In [2]:
import pandas as pd

In [3]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from torchgeo_bench.coordbench.datasets import (
    load_daily_usa_electric_usage, load_era5_ecmwf, load_chelsa_monthly)

In [4]:
def decompose(b, task):
    df = pd.DataFrame({"county": b.spatial_aggregation_key[1].to_numpy(),
                       "t": b.posix_timestamp, "y": b.tasks[task]}).dropna()
    df = df[np.isfinite(df.y)]
    y = df.y - df.y.mean()
    tot = (y**2).sum()
    cm = df.groupby("county").y.transform("mean") - df.y.mean()   # spatial part
    tm = df.groupby("t").y.transform("mean") - df.y.mean()        # shared temporal part
    resid = y - cm - tm                                            # county-specific temporal part
    return pd.Series({
        "R2_space (county means)": (cm**2).sum() / tot,
        "R2_time (day means)":     (tm**2).sum() / tot,
        "R2_space+time":           1 - (resid**2).sum() / tot,
        "median within-county CV": (df.groupby("county").y.std() / df.groupby("county").y.mean()).median(),
    })

In [5]:
elec = load_daily_usa_electric_usage(subsample_locations=True)[0]

/home/libe2152/projects/torchgeo-bench/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
pd.DataFrame({t: decompose(elec, t) for t in elec.tasks}).T.round(3)

,R2_space (county means),R2_time (day means),R2_space+time,median within-county CV
median,0.967,0.003,0.968,0.138
median_log,0.987,0.006,0.991,0.035
